In [1]:
# core
import os
import argparse
import copy
import pickle
import random
# numerical / data
import numpy as np
import pandas as pd

# torch
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import datasets, models, transforms
from torchvision.datasets import STL10, ImageFolder
from torchvision.transforms import ToTensor, functional as TF

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
# plotting
import matplotlib.pyplot as plt
import seaborn as sns
# misc
from collections import Counter
from sklearn.metrics import ConfusionMatrixDisplay

import torch
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
#print("device name:", torch.cuda.get_device_name(0))

import wandb
from ultralytics import YOLO
from IPython.display import Image, display
import glob
from pathlib import Path
import kagglehub, os, shutil
import yaml as pyyaml 
import xml.etree.ElementTree as ET

random.seed(42)

cuda available: True
device count: 1


In [7]:
import wandb
print(wandb.__version__)

# login for weights and biases 
wandb.login(key="")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /users/jmatthia/.netrc


0.25.0


True

# Creating the correct file format for YOLOv3

In [4]:
import yaml
yaml_path = Path("/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml")
data = yaml.safe_load(open(yaml_path))

print("Keys:", data.keys())
print("path:", data.get("path"))
print("train:", data.get("train"))
print("val:", data.get("val"))
print("nc:", data.get("nc"))
print("names:", data.get("names")[:5] if isinstance(data.get("names"), list) else data.get("names"))


Keys: dict_keys(['path', 'train', 'val', 'test', 'names'])
path: /users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split
train: images/train
val: images/val
nc: None
names: {0: 'aeroplane', 1: 'bicycle', 2: 'bird', 3: 'boat', 4: 'bottle', 5: 'bus', 6: 'car', 7: 'cat', 8: 'chair', 9: 'cow', 10: 'diningtable', 11: 'dog', 12: 'horse', 13: 'motorbike', 14: 'person', 15: 'pottedplant', 16: 'sheep', 17: 'sofa', 18: 'train', 19: 'tvmonitor'}


In [11]:
import yaml
from pathlib import Path

in_yaml = Path("/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml")
out_yaml = in_yaml.with_name("voc_yolov3.yaml")

d = yaml.safe_load(open(in_yaml))

# convert dict {0:'a',1:'b',...} -> list ['a','b',...]
names = d["names"]
if isinstance(names, dict):
    names_list = [names[i] for i in range(len(names))]
else:
    names_list = list(names)

d["names"] = names_list
d["nc"] = len(names_list)

with open(out_yaml, "w") as f:
    yaml.safe_dump(d, f, sort_keys=False)

print("Wrote:", out_yaml)
print("nc:", d["nc"])
print("names[0:5]:", d["names"][:5])

Wrote: /users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc_yolov3.yaml
nc: 20
names[0:5]: ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle']


# Training YOLOv3

Downaloded DarkNet53 weights from here:! wget -O darknet53.conv.74 https://sourceforge.net/projects/yolov3.mirror/files/v8/darknet53.conv.74/download

In [18]:
!cd /users/jmatthia/deep_learning/code/assignment_3/yolov3
!grep -R "load_darknet" -n .

In [12]:
%cd /users/jmatthia/deep_learning/code/assignment_3/yolov3

!python train.py \
  --img 640 \
  --batch-size 16 \
  --epochs 450 \
  --data /users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc_yolov3.yaml \
  --cfg models/yolov3.yaml \
  --weights darknet53.conv.74 \
  --name yolov3_voc_darknet53_imagenet 2>&1 | tee train_log.txt

/users/jmatthia/deep_learning/code/assignment_3/yolov3


wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /users/jmatthia/.netrc.
wandb: Currently logged in as: jan-matthias (jan-matthias-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
train: weights=darknet53.conv.74, cfg=models/yolov3.yaml, data=/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc_yolov3.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=450, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, project=runs/train, name=yolov3_voc_darknet53_imagenet, exist_ok=False, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, fre